# Stage 04 - Build Streaming Measures

Calculate freshness, continuity, sequence, acknowledgment, and contract-health measures from the shared simulated event stream.

In [ ]:
from datetime import timedelta
import json
from pyspark.sql import Window, functions as F

bronze_events_df = spark.table('bronze_recorded_observed_events')

rules_path = 'Files/shared/integrated-test-data/projections/realtime/finding_rules.json'
rules_text = '\n'.join(row.value for row in spark.read.text(rules_path).collect())
rules = json.loads(rules_text)

table_names = {
    'measures': 'silver_streaming_measures',
    'issues': 'silver_stream_contract_issues',
    'expectations': 'silver_stream_expectations',
}

freshness_threshold_seconds = rules['freshness_lag_seconds']
continuity_gap_seconds = rules['continuity_gap_seconds']
max_sequence_gap = rules['max_sequence_gap']
assignment_ack_due_seconds = rules['assignment_ack_due_seconds']
uuid_pattern = '^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-4[0-9a-fA-F]{3}-8[0-9a-fA-F]{3}-[0-9a-fA-F]{12}$'
allowed_source_systems = ['TPY2', 'PATRIOT', 'THAAD', 'BATTLE_MANAGEMENT', 'ARMY_INTEGRATION', 'SCENARIO_CONTROLLER', 'SUSTAINMENT']


In [ ]:
staged_events_df = (
    bronze_events_df
    .withColumn('event_time_ts', F.to_timestamp('event_time_utc'))
    .withColumn('ingest_time_ts', F.to_timestamp('ingest_time_utc'))
    .withColumn('site_id_normalized', F.when(F.trim(F.coalesce(F.col('site_id'), F.lit(''))) == '', F.lit(None)).otherwise(F.col('site_id')))
    .withColumn('event_id_is_valid', F.col('event_id').rlike(uuid_pattern))
    .withColumn('site_id_is_valid', F.col('site_id_normalized').isNotNull())
    .withColumn('source_system_is_known', F.col('source_system').isin(allowed_source_systems))
    .withColumn('classification_is_valid', F.col('classification') == F.lit(rules['classification_required']))
    .withColumn('event_time_is_valid', F.col('event_time_ts').isNotNull())
    .withColumn('ingest_time_is_valid', F.col('ingest_time_ts').isNotNull())
    .withColumn(
        'schema_issue',
        F.when(~F.col('event_id_is_valid'), F.lit('INVALID_EVENT_ID'))
        .when(~F.col('event_time_is_valid'), F.lit('INVALID_EVENT_TIME'))
        .when(~F.col('ingest_time_is_valid'), F.lit('INVALID_INGEST_TIME'))
        .when(~F.col('site_id_is_valid'), F.lit('MISSING_SITE_ID'))
        .when(~F.col('source_system_is_known'), F.lit('UNKNOWN_SOURCE_SYSTEM'))
        .when(~F.col('classification_is_valid'), F.lit('INVALID_CLASSIFICATION'))
    )
)

dedupe_window = Window.partitionBy('event_id').orderBy(F.col('ingest_time_ts').asc_nulls_last(), F.col('bronze_record_sha256').asc())
ranked_events_df = staged_events_df.withColumn('duplicate_rank', F.row_number().over(dedupe_window))

canonical_events_df = ranked_events_df.filter(F.col('schema_issue').isNull() & (F.col('duplicate_rank') == 1))

contract_issue_payload = [F.col(column_name) for column_name in ranked_events_df.columns]
contract_issues_df = (
    ranked_events_df
    .filter(F.col('schema_issue').isNotNull() | (F.col('duplicate_rank') > 1))
    .withColumn('issue_type', F.when(F.col('schema_issue').isNotNull(), F.col('schema_issue')).otherwise(F.lit('DUPLICATE_EVENT_ID')))
    .select(
        F.col('event_id').alias('record_id'),
        'issue_type',
        'event_type',
        'source_system',
        'source_instance_id',
        'ordering_key',
        'correlation_id',
        'target_instance_id',
        'sequence_number',
        'duplicate_rank',
        'event_time_utc',
        'ingest_time_utc',
        'event_time_ts',
        'ingest_time_ts',
        'bronze_record_sha256',
        F.to_json(F.struct(*contract_issue_payload)).alias('raw_payload_json'),
    )
)

print(f'Canonical replay events: {canonical_events_df.count()}')
print(f'Contract issues: {contract_issues_df.count()}')


In [ ]:
event_window = Window.partitionBy('ordering_key').orderBy(F.col('event_time_ts').asc_nulls_last(), F.col('sequence_number').asc())

measures_df = (
    canonical_events_df
    .withColumn('previous_event_id', F.lag('event_id').over(event_window))
    .withColumn('previous_sequence_number', F.lag('sequence_number').over(event_window))
    .withColumn('previous_event_time_ts', F.lag('event_time_ts').over(event_window))
    .withColumn('ingest_lag_seconds', F.unix_timestamp('ingest_time_ts') - F.unix_timestamp('event_time_ts'))
    .withColumn('sequence_gap_count', F.greatest(F.coalesce(F.col('sequence_number') - F.col('previous_sequence_number') - F.lit(1), F.lit(0)), F.lit(0)))
    .withColumn('continuity_gap_seconds', F.greatest(F.coalesce(F.unix_timestamp('event_time_ts') - F.unix_timestamp('previous_event_time_ts'), F.lit(0)), F.lit(0)))
    .withColumn('quality_score', F.when(F.col('event_type') == 'sensor.observation', F.col('quality.score')).otherwise(F.lit(None).cast('double')))
    .withColumn('quality_freshness_ms', F.when(F.col('event_type') == 'sensor.observation', F.col('quality.freshness_ms')).otherwise(F.lit(None).cast('double')))
    .withColumn('quality_continuity_score', F.when(F.col('event_type') == 'sensor.observation', F.col('quality.continuity')).otherwise(F.lit(None).cast('double')))
    .withColumn('command_processing_delay_ms', F.when(F.col('event_type') == 'command.integration', F.col('processing_delay_ms')).otherwise(F.lit(None).cast('double')))
    .withColumn(
        'readiness_or_marker_state',
        F.when(F.col('event_type') == 'system.status', F.col('readiness_state'))
        .when(F.col('event_type').isin('test.marker', 'preservation.marker'), F.col('marker_status'))
        .otherwise(F.col('status'))
    )
    .withColumn('is_late_event', F.col('ingest_lag_seconds') > F.lit(freshness_threshold_seconds))
    .withColumn('has_sequence_gap', F.col('sequence_gap_count') > F.lit(max_sequence_gap))
    .withColumn('has_continuity_gap', F.col('continuity_gap_seconds') > F.lit(continuity_gap_seconds))
    .withColumn('low_continuity_signal', F.when(F.col('quality_continuity_score').isNotNull(), F.col('quality_continuity_score') < F.lit(0.8)).otherwise(F.lit(False)))
    .withColumn('timing_rule_exceeded', F.when(F.col('command_processing_delay_ms').isNotNull(), F.col('command_processing_delay_ms') > F.lit(rules['command_processing_delay_ms'])).otherwise(F.lit(False)))
    .withColumn('schema_valid', F.lit(True))
    .select(
        'event_id',
        'event_type',
        'source_system',
        'source_instance_id',
        'ordering_key',
        'sequence_number',
        'previous_event_id',
        'previous_sequence_number',
        'event_time_utc',
        'ingest_time_utc',
        'event_time_ts',
        'ingest_time_ts',
        'ingest_lag_seconds',
        'sequence_gap_count',
        'continuity_gap_seconds',
        'quality_score',
        'quality_freshness_ms',
        'quality_continuity_score',
        'command_processing_delay_ms',
        'readiness_or_marker_state',
        'is_late_event',
        'has_sequence_gap',
        'has_continuity_gap',
        'low_continuity_signal',
        'timing_rule_exceeded',
        'schema_valid',
        'track_id',
        'correlation_id',
        'target_instance_id',
        'marker_name',
        'bronze_record_sha256',
    )
)

print(f'Streaming measure rows: {measures_df.count()}')


In [ ]:
marker_lookup = {
    row['marker_name']: row
    for row in canonical_events_df.filter(F.col('event_type').isin('test.marker', 'preservation.marker')).select('event_id', 'marker_name', 'event_time_ts').collect()
}

expectation_rows = []
for marker_rule in rules['expected_markers']:
    source_row = marker_lookup.get(marker_rule['marker_name'])
    if 'must_precede' in marker_rule:
        expected_name = marker_rule['must_precede']
        target_row = marker_lookup.get(expected_name)
        status = 'MET' if source_row and target_row and source_row['event_time_ts'] <= target_row['event_time_ts'] else 'MISSING_OR_OUT_OF_ORDER'
        evidence_ids = [value for value in [source_row['event_id'] if source_row else None, target_row['event_id'] if target_row else None] if value]
        expectation_rows.append((f"marker::{marker_rule['marker_name']}::{expected_name}", 'MARKER_ORDER', marker_rule['marker_name'], expected_name, source_row['event_id'] if source_row else None, target_row['event_id'] if target_row else None, target_row['event_time_ts'] if target_row else None, status, evidence_ids, None, None))
    if 'must_follow' in marker_rule:
        expected_name = marker_rule['must_follow']
        target_row = marker_lookup.get(expected_name)
        status = 'MET' if source_row and target_row and source_row['event_time_ts'] >= target_row['event_time_ts'] else 'MISSING_OR_OUT_OF_ORDER'
        evidence_ids = [value for value in [source_row['event_id'] if source_row else None, target_row['event_id'] if target_row else None] if value]
        expectation_rows.append((f"marker::{marker_rule['marker_name']}::{expected_name}", 'MARKER_ORDER', marker_rule['marker_name'], expected_name, source_row['event_id'] if source_row else None, target_row['event_id'] if target_row else None, source_row['event_time_ts'] if source_row else None, status, evidence_ids, None, None))

observation_window_end = canonical_events_df.agg(F.max('event_time_ts').alias('window_end')).first()['window_end']
integration_rows = canonical_events_df.filter(F.col('event_type') == 'command.integration').select('event_id', 'action', 'correlation_id', 'target_instance_id', 'event_time_ts').collect()
ack_lookup = {
    (row['correlation_id'], row['target_instance_id']): row
    for row in integration_rows
    if row['action'] == 'ASSIGNMENT_ACKNOWLEDGED'
}

for row in integration_rows:
    if row['action'] == 'ABSTRACT_ASSIGNMENT':
        ack_row = ack_lookup.get((row['correlation_id'], row['target_instance_id']))
        due_by = row['event_time_ts'] + timedelta(seconds=assignment_ack_due_seconds)
        if ack_row:
            status = 'MET' if ack_row['event_time_ts'] <= due_by else 'LATE'
        else:
            status = 'OVERDUE' if observation_window_end and observation_window_end >= due_by else 'UNDETERMINED'
        evidence_ids = [value for value in [row['event_id'], ack_row['event_id'] if ack_row else None] if value]
        expectation_rows.append((f"follow_on::{row['event_id']}", 'FOLLOW_ON_EVENT', row['action'], 'ASSIGNMENT_ACKNOWLEDGED', row['event_id'], ack_row['event_id'] if ack_row else None, due_by, status, evidence_ids, row['correlation_id'], row['target_instance_id']))

expectations_df = spark.createDataFrame(
    expectation_rows,
    [
        'expectation_id',
        'expectation_type',
        'trigger_name',
        'expected_name',
        'source_event_id',
        'actual_event_id',
        'due_by_utc',
        'expectation_status',
        'evidence_event_ids',
        'correlation_id',
        'target_instance_id',
    ],
)

for table_name, frame in {
    table_names['measures']: measures_df,
    table_names['issues']: contract_issues_df,
    table_names['expectations']: expectations_df,
}.items():
    (
        frame.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(table_name)
    )

spark.table(table_names['issues']).orderBy('issue_type', 'record_id').show(truncate=False)
spark.table(table_names['measures']).select('event_id', 'event_type', 'source_system', 'ingest_lag_seconds', 'sequence_gap_count', 'continuity_gap_seconds', 'is_late_event', 'has_sequence_gap', 'has_continuity_gap', 'timing_rule_exceeded').orderBy('event_time_ts').show(truncate=False)
spark.table(table_names['expectations']).orderBy('expectation_type', 'expectation_id').show(truncate=False)